# Mario PPO · điểm số và đa màn

Kiểm thử đơn vị đã chạy trên CPU; mã này chưa được dùng cho lần train mới nào.
Chạy từng phần sau khi kiểm tra tài nguyên.

In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys, json, hashlib, tempfile, zipfile, subprocess

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/Mario')
DRIVE_RUNS = DRIVE_ROOT / 'runs'
SOURCE_ZIP = DRIVE_ROOT / 'mario_upgrade_v2.zip'
if any(name == 'mario_rl' or name.startswith('mario_rl.') for name in sys.modules):
    raise RuntimeError('Restart runtime trước khi đổi gói mã nguồn.')
with zipfile.ZipFile(SOURCE_ZIP) as archive:
    manifest = json.loads(archive.read('MANIFEST.json'))
    names = archive.namelist()
    if len(names) != len(set(names)) or set(names) != set(manifest) | {'MANIFEST.json'}:
        raise ValueError('ZIP không khớp manifest')
    for name, item in manifest.items():
        path = Path(name)
        if path.is_absolute() or '..' in path.parts or '\\' in name or ':' in name:
            raise ValueError(f'Đường dẫn không hợp lệ: {name}')
        data = archive.read(name)
        if len(data) != item['bytes'] or hashlib.sha256(data).hexdigest() != item['sha256']:
            raise ValueError(f'Checksum sai: {name}')
    WORK_DIR = Path(tempfile.mkdtemp(prefix='mario_v2_', dir='/content'))
    archive.extractall(WORK_DIR)
os.chdir(WORK_DIR)
sys.path.insert(0, str(WORK_DIR))
print(WORK_DIR)


In [ ]:
from importlib.metadata import version, PackageNotFoundError

INSTALL_MISSING = True   # False: chỉ kiểm tra, không tự cài
required = {'gym': '0.26.2', 'nes-py': '8.2.1', 'gym-super-mario-bros': '7.4.0'}


def installed(package):
    try:
        return version(package)
    except PackageNotFoundError:
        return None


missing = [p for p in required if installed(p) is None]
if missing and INSTALL_MISSING:
    # nes-py và gym-super-mario-bros khai báo phụ thuộc cũ nên cài --no-deps; không nâng cấp gói có sẵn.
    for package in missing:
        extra = [] if package == 'gym' else ['--no-deps']
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                        f'{package}=={required[package]}', *extra], check=True)
for package, expected in required.items():
    actual = installed(package)
    if actual is None:
        raise RuntimeError(f'Thiếu {package}=={expected}; đặt INSTALL_MISSING = True hoặc xem docs/DRIVE_START_HERE.md')
    if actual != expected:
        raise RuntimeError(f'{package}: cần {expected}, hiện có {actual}')
# Vá nes-py cho NumPy 2; chạy lại nhiều lần không sao.
subprocess.run([sys.executable, 'scripts/patch_nes_py_numpy2.py'], check=True)
for package in ('torch', 'numpy'):
    print(package, version(package))

## Cấu hình

In [ ]:
PROFILE = 'score_1-1'      # 'multistage' sau khi đánh giá giai đoạn 1
RUN_NAME = None             # None: dùng tên trong profile; đổi tên cho thí nghiệm mới
RESUME = False
TOTAL_STEPS = None          # khi resume: đặt tổng bước lớn hơn checkpoint
DEVICE = 'cpu'             # chủ động đổi sang 'cuda' nếu muốn dùng GPU
SEED = 1

from mario_rl.drive_backup import restore_run
cfg = json.loads((WORK_DIR / 'configs' / f'{PROFILE}.json').read_text())
run_name = RUN_NAME or cfg['run_name']
if Path(run_name).name != run_name or run_name in ('.', '..'):
    raise ValueError('RUN_NAME phải là một tên thư mục')
local_run = WORK_DIR / 'runs' / run_name
drive_run = DRIVE_RUNS / run_name
if RESUME:
    if not local_run.exists() or not any(local_run.iterdir()):
        restore_run(drive_run, local_run)
    cfg = json.loads((local_run / 'config.json').read_text())
    cfg.update(resume=True, init_from='')
else:
    if drive_run.exists() and any(drive_run.iterdir()):
        raise FileExistsError('Run đã tồn tại trên Drive; đổi tên hoặc dùng RESUME.')
    if local_run.exists() and any(local_run.iterdir()):
        raise FileExistsError('Run đã tồn tại cục bộ; đổi tên hoặc dùng RESUME.')
    source_run = Path(cfg['init_from']).parent.name
    source = DRIVE_ROOT / 'checkpoints' / source_run / 'ckpt_final.pt'
    if not source.is_file():
        source_local = WORK_DIR / 'transfer_sources' / source_run
        if not source_local.exists():
            restore_run(DRIVE_RUNS / source_run, source_local)
        source = source_local / 'ckpt_final.pt'
        if not source.is_file():
            raise FileNotFoundError(f'Chưa có checkpoint final của {source_run}: {source}')
    cfg.update(init_from=str(source), seed=SEED)
cfg.update(run_name=run_name, out_dir=str(WORK_DIR / 'runs'), device=DEVICE,
           drive_backup_dir=str(drive_run), backup_interval=300)
if TOTAL_STEPS is not None:
    cfg['total_steps'] = int(TOTAL_STEPS)
CONFIG_PATH = WORK_DIR / 'experiment.json'
CONFIG_PATH.write_text(json.dumps(cfg, indent=2))
print(json.dumps(cfg, indent=2))


## Huấn luyện

In [ ]:
# Foreground: chỉ chuyển sang đánh giá khi lệnh này kết thúc.
subprocess.run([sys.executable, '-u', '-m', 'mario_rl.cli.train',
                '--config', str(CONFIG_PATH)], check=True)


## Đánh giá

In [ ]:
from datetime import datetime

CHECKPOINT = local_run / 'ckpt_final.pt'
BASELINE = DRIVE_ROOT / 'checkpoints' / 'ppo_1-1' / 'ckpt_final.pt'
EVAL_STAGES = '1-1'  # kiểm tra giai đoạn 1; mở rộng danh sách khi sẵn sàng
EVAL_SEEDS = '12345,23456,34567'
EPISODES = 20       # mỗi màn, seed và chế độ
ckpts = [str(CHECKPOINT)]
if BASELINE.is_file():
    ckpts.insert(0, str(BASELINE))
else:
    print('Chưa có baseline cũ tại', BASELINE, '— chưa thể so sánh trước/sau.')
EVAL_OUT = DRIVE_ROOT / 'evaluations' / (run_name + '-' + datetime.now().strftime('%Y%m%d-%H%M%S'))
subprocess.run([sys.executable, '-u', '-m', 'mario_rl.cli.evaluate_suite',
               '--ckpts', *ckpts, '--stages', EVAL_STAGES, '--seeds', EVAL_SEEDS,
               '--episodes', str(EPISODES), '--noop_max', '30', '--num_envs', '4',
               '--device', DEVICE, '--out', str(EVAL_OUT)], check=True)


In [ ]:
import pandas as pd

table = pd.read_csv(EVAL_OUT / 'summary.csv')
display(table[['checkpoint', 'stage', 'trained_stage', 'mode', 'seed',
               'flag_rate', 'score_gain_mean', 'coins_collected_mean', 'max_x_mean']])


## Demo

In [ ]:
from IPython.display import Image, display

video_dir = DRIVE_ROOT / 'demos' / run_name
video_dir.mkdir(parents=True, exist_ok=True)
gif = video_dir / ('1-1-' + datetime.now().strftime('%Y%m%d-%H%M%S') + '.gif')
subprocess.run([sys.executable, '-m', 'mario_rl.cli.evaluate',
               '--ckpt', str(CHECKPOINT), '--stage', '1-1', '--episodes', '1',
               '--device', DEVICE, '--gif', str(gif)], check=True)
display(Image(filename=str(gif)))
